# 11 — Constraint Score Improvements + Full POI Enrichment

- **D1** SFA API fix
- **D2** SFA fit on outlet-level data (~20k rows) instead of 450k transaction rows — runs in seconds
- **D3** Full-country POI enrichment (20,000/20,000 coverage via `10_poi_enrichment_full.ipynb`)
- **D4** `s_poi` as direct 5th constraint score component (0.10 weight)
- **D5** POI ceiling boost: peer soft cap scaled by `1.0 + 0.10 × poi_demand_score` (bounded by hard ratio cap)

Inherited fixes from `05`:
- **B1** `Outlet_ID` column, 20 000 rows
- **B2** `same_type_outlet_count_2km` → cannibalisation (sign flipped)
- **B3** Distributor-median coordinate imputation
- **B4** `has_valid_coordinates` removed from constraint score
- **C1–C6** Composite constraint score, SFA, plateau detector, enriched features, cap 4.0×, no `^1.25`

**Outputs:**
- `Results/smile_labs_predictions.csv` — base predictions (sparse POI, 4-component score)
- `Results/smile_labs_predictions_poi_full.csv` — full POI predictions (5-component score)

## 1. Setup

In [ ]:
from __future__ import annotations
import subprocess, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import QuantileRegressor
from sklearn.neighbors import BallTree
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
warnings.filterwarnings('ignore')
try:
    pd.options.future.infer_string = False
except AttributeError:
    pass

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
SILVER_DIR   = PROJECT_ROOT / 'data' / 'silver'
GOLD_DIR     = PROJECT_ROOT / 'data' / 'gold'
RESULTS_DIR  = PROJECT_ROOT / 'Results'
RESULTS_DIR.mkdir(exist_ok=True)

SIZE_SCORE  = {'Unknown': 0, 'Small': 1, 'Medium': 2, 'Large': 3, 'Extra Large': 4}
TYPE_SCORE  = {'Kiosk': 0.80, 'Pharmacy': 0.85, 'Bakery': 1.00, 'Grocery': 1.10, 'Eatery': 1.15, 'Hotel': 1.20, 'SMMT': 1.25}
TYPE_MAP    = {'Grocry': 'Grocery', 'Bakry': 'Bakery', 'Eatery ': 'Eatery'}
SIZE_MAP    = {'small': 'Small', '': 'Unknown'}
SEA_SCORE   = {'Un-Favorable': -1, 'Moderate': 0, 'Favorable': 1}
RANDOM_STATE = 2026
EARTH_KM    = 6371.0088
REF_YEAR, REF_MONTH = 2025, 12

SFA_AVAILABLE = False
try:
    import pysfa; SFA_AVAILABLE = True; print('pySFA ready')
except ImportError:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'pysfa', '-q'], check=True, capture_output=True)
        import pysfa; SFA_AVAILABLE = True; print('pySFA installed')
    except Exception as e:
        print(f'pySFA unavailable ({e}); s_sfa=0.5 (neutral fallback)')

print(f'Root: {PROJECT_ROOT}  |  pandas {pd.__version__}  |  SFA={SFA_AVAILABLE}')

## 2. Load Silver Data

In [ ]:
def _ns(s): return s.fillna('').astype(str).str.strip()

outlets      = pd.read_csv(SILVER_DIR / 'outlet_master.csv')
coords_raw   = pd.read_csv(SILVER_DIR / 'outlet_coordinates.csv')
transactions = pd.read_csv(SILVER_DIR / 'transactions_history.csv')
seasonality  = pd.read_csv(SILVER_DIR / 'distributor_seasonality.csv')
holidays     = pd.read_csv(SILVER_DIR / 'holiday_list.csv')

outlets['Outlet_ID']     = outlets['Outlet_ID'].astype(str)
outlets['Cooler_Count']  = pd.to_numeric(outlets['Cooler_Count'], errors='coerce').fillna(0)
outlets['Outlet_Size']   = _ns(outlets['Outlet_Size']).replace(SIZE_MAP).replace('', 'Unknown')
outlets['Outlet_Type']   = _ns(outlets['Outlet_Type']).replace(TYPE_MAP)

coords_raw['Outlet_ID']  = coords_raw['Outlet_ID'].astype(str)
coords_raw['Latitude']   = pd.to_numeric(coords_raw['Latitude'],  errors='coerce')
coords_raw['Longitude']  = pd.to_numeric(coords_raw['Longitude'], errors='coerce')
coords_raw['valid']      = coords_raw['Latitude'].between(5.5,10.2) & coords_raw['Longitude'].between(79.0,82.1)
coords_raw.loc[~coords_raw['valid'], ['Latitude','Longitude']] = np.nan

for col in ['Outlet_ID','Distributor_ID','SKU_ID']:
    if col in transactions.columns:
        transactions[col] = transactions[col].astype(str)
for col in ['Year','Month','Volume_Liters']:
    transactions[col] = pd.to_numeric(transactions[col], errors='coerce')

seasonality['Distributor_ID']    = seasonality['Distributor_ID'].astype(str)
seasonality['Seasonality_Score'] = _ns(seasonality['Seasonality_Index']).map(SEA_SCORE).fillna(0)
seasonality['Month']             = pd.to_numeric(seasonality['Month'], errors='coerce')
holidays['Date']  = pd.to_datetime(holidays['Date'], errors='coerce', utc=True)
holidays['Month'] = holidays['Date'].dt.month

mp = GOLD_DIR / 'outlet_monthly_sales.csv'
if mp.exists():
    monthly = pd.read_csv(mp)
    for c in ['Outlet_ID','Distributor_ID']: monthly[c] = monthly[c].astype(str)
    print(f'Loaded monthly sales: {len(monthly):,}')
else:
    monthly = transactions.groupby(['Outlet_ID','Year','Month'], as_index=False).agg(
        Monthly_Liters=('Volume_Liters','sum'), SKU_Count=('SKU_ID','nunique'),
        Transaction_Lines=('SKU_ID','size'),
        Distributor_ID=('Distributor_ID', lambda x: x.mode().iat[0]))
    for c in ['Outlet_ID','Distributor_ID']: monthly[c] = monthly[c].astype(str)
    monthly.to_csv(mp, index=False)
    print(f'Built monthly sales: {len(monthly):,}')

## 3. Gold Feature Engineering (all bug fixes applied)

In [ ]:
sa = monthly.groupby('Outlet_ID').agg(
    observed_mean_monthly_liters   =('Monthly_Liters','mean'),
    observed_median_monthly_liters =('Monthly_Liters','median'),
    observed_max_monthly_liters    =('Monthly_Liters','max'),
    observed_std_monthly_liters    =('Monthly_Liters','std'),
    active_months                  =('Monthly_Liters','size'),
    mean_sku_count                 =('SKU_Count','mean'),
    max_sku_count                  =('SKU_Count','max'),
    mean_transaction_lines         =('Transaction_Lines','mean'),
).reset_index()
jan    = monthly[monthly['Month'].astype(int)==1].groupby('Outlet_ID').agg(
    january_mean_liters=('Monthly_Liters','mean'), january_max_liters=('Monthly_Liters','max')).reset_index()
recent = monthly[(monthly['Year'].astype(int)==2025) & monthly['Month'].astype(int).isin([10,11,12])].groupby('Outlet_ID').agg(
    recent_3_month_mean_liters=('Monthly_Liters','mean'), recent_3_month_max_liters=('Monthly_Liters','max')).reset_index()
dom_dist = (monthly.groupby(['Outlet_ID','Distributor_ID']).size().rename('n').reset_index()
            .sort_values(['Outlet_ID','n'],ascending=[True,False]).drop_duplicates('Outlet_ID')[['Outlet_ID','Distributor_ID']])

feat = outlets.copy()
for df in [sa, jan, recent, dom_dist]: feat = feat.merge(df, on='Outlet_ID', how='left')
feat['Distributor_ID'] = feat['Distributor_ID'].fillna('UNKNOWN')

# B3: distributor-median coordinate imputation
feat = feat.merge(coords_raw[['Outlet_ID','Latitude','Longitude','valid']].rename(columns={'valid':'has_valid_coordinates'}), on='Outlet_ID', how='left')
feat['has_valid_coordinates'] = feat['has_valid_coordinates'].fillna(False).astype(int)
for col in ['Latitude','Longitude']:
    feat[col] = feat[col].fillna(feat.groupby('Distributor_ID')[col].transform('median'))
    feat[col] = feat[col].fillna(feat[col].median())
print(f"Imputed coords: {(feat['has_valid_coordinates']==0).sum()} outlets via distributor-median")

# BallTree catchment + B2: same-type = cannibalisation (subtracted)
for c in ['outlet_count_1km','outlet_count_2km','outlet_count_5km','same_type_cannibalisation_2km','nearest_outlet_distance_km','catchment_density_score']:
    feat[c] = 0.0
vm  = feat['has_valid_coordinates'].eq(1)
vg  = feat.loc[vm, ['Latitude','Longitude','Outlet_Type']].copy()
vidx = vg.index.to_numpy()
if len(vidx) > 1:
    cr = np.radians(vg[['Latitude','Longitude']].to_numpy())
    bt = BallTree(cr, metric='haversine')
    for km, col in [(1,'outlet_count_1km'),(2,'outlet_count_2km'),(5,'outlet_count_5km')]:
        feat.loc[vidx, col] = (bt.query_radius(cr, r=km/EARTH_KM, count_only=True)-1).astype(float)
    nd,_ = bt.query(cr, k=2)
    feat.loc[vidx,'nearest_outlet_distance_km'] = nd[:,1]*EARTH_KM
    feat['nearest_outlet_distance_km'].fillna(feat['nearest_outlet_distance_km'].max(), inplace=True)
    for ot, gi in vg.groupby('Outlet_Type').groups.items():
        gi = np.array(list(gi))
        if len(gi)>1:
            gr = np.radians(feat.loc[gi,['Latitude','Longitude']].to_numpy())
            gt = BallTree(gr, metric='haversine')
            feat.loc[gi,'same_type_cannibalisation_2km'] = (gt.query_radius(gr,r=2/EARTH_KM,count_only=True)-1).astype(float)
    ns = 1 - feat['nearest_outlet_distance_km'].rank(pct=True)
    feat['catchment_density_score'] = (
        0.40*feat['outlet_count_1km'].rank(pct=True)
        + 0.25*feat['outlet_count_2km'].rank(pct=True)
        + 0.20*feat['outlet_count_5km'].rank(pct=True)
        + 0.15*ns.fillna(0)
        - 0.15*feat['same_type_cannibalisation_2km'].rank(pct=True)  # cannibalisation penalty
    ).clip(0,1)

# Seasonality, holidays
js = seasonality[seasonality['Month']==1].groupby('Distributor_ID',as_index=False)['Seasonality_Score'].mean().rename(columns={'Seasonality_Score':'target_jan_seasonality_score'})
feat = feat.merge(js, on='Distributor_ID', how='left')
feat['target_jan_holiday_count'] = float(holidays[holidays['Month']==1].shape[0])

# POI
poi_path = GOLD_DIR / 'outlet_poi_features.csv'
poi_cols = ['poi_total_count_1km','poi_total_count_2km','poi_demand_score']
if poi_path.exists():
    poi = pd.read_csv(poi_path); poi['Outlet_ID'] = poi['Outlet_ID'].astype(str)
    feat = feat.merge(poi[['Outlet_ID']+poi_cols], on='Outlet_ID', how='left')
    print(f"POI coverage: {feat['poi_demand_score'].notna().sum():,}/{len(feat):,}")
for c in poi_cols: feat[c] = pd.to_numeric(feat.get(c,0),errors='coerce').fillna(0)

# Structural & enriched features
feat['Outlet_Size_Score']        = feat['Outlet_Size'].map(SIZE_SCORE).fillna(0)
feat['Outlet_Type_Score']        = feat['Outlet_Type'].map(TYPE_SCORE).fillna(1.0)
feat['structural_capacity_score']= (1+0.18*feat['Outlet_Size_Score']+0.10*feat['Cooler_Count'])*feat['Outlet_Type_Score']
feat['cv_liters']                = (feat['observed_std_monthly_liters']/(feat['observed_mean_monthly_liters']+1)).fillna(0)
feat['trend_ratio']              = (feat['recent_3_month_mean_liters']/(feat['observed_mean_monthly_liters']+1)).fillna(1.0).clip(0,5)
feat['peer_pct_by_type_size']    = feat.groupby(['Outlet_Type','Outlet_Size'])['observed_max_monthly_liters'].rank(pct=True).fillna(0.5)
feat['peer_pct_by_size']         = feat.groupby('Outlet_Size')['observed_max_monthly_liters'].rank(pct=True).fillna(0.5)
pmc = feat.groupby(['Outlet_Type','Outlet_Size'])['Cooler_Count'].transform('median').fillna(0)
feat['cooler_adequacy']          = feat['Cooler_Count']/(pmc+1)
feat['sku_consistency']          = (feat['mean_sku_count']/(feat['max_sku_count']+1)).fillna(0)
feat['volume_per_cooler']        = feat['observed_max_monthly_liters']/(feat['Cooler_Count']+1)
feat['volume_per_sku']           = feat['observed_max_monthly_liters']/(feat['mean_sku_count']+1)
_ds = monthly.groupby('Distributor_ID')['Monthly_Liters'].mean().rename('dist_monthly_strength').reset_index()
_ds['Distributor_ID'] = _ds['Distributor_ID'].astype(str)
feat = feat.merge(_ds, on='Distributor_ID', how='left')
feat[feat.select_dtypes(include=[np.number]).columns] = feat.select_dtypes(include=[np.number]).fillna(0)

feat.to_csv(GOLD_DIR/'outlet_features_v2.csv', index=False)
print(f'Feature matrix: {feat.shape}')
feat.head(3)

## 4. Plateau Detection
Variance collapse + months-since-new-max (research/07 §3).

In [ ]:
monthly['month_idx'] = (monthly['Year'].astype(int)-2023)*12 + monthly['Month'].astype(int)
ref_idx = (REF_YEAR-2023)*12 + REF_MONTH
ms = monthly.sort_values(['Outlet_ID','month_idx'])

def _vr(g):
    v = g['Monthly_Liters'].values
    if len(v)<6: return 0.5
    return float(np.var(v[-6:]) / (np.var(v[-12:-6]) if len(v)>=12 else 1e-9) + 1e-9)

def _msm(g):
    v,mi = g['Monthly_Liters'].values, g['month_idx'].values
    if len(v)==0: return 12.0
    rmax, lm = -np.inf, mi[0]
    for vol,midx in zip(v,mi):
        if vol >= rmax-0.01: rmax,lm = vol,midx
    return float(ref_idx - lm)

vr  = ms.groupby('Outlet_ID').apply(_vr).rename('var_ratio').reset_index()
msm = ms.groupby('Outlet_ID').apply(_msm).rename('months_since_new_max').reset_index()
feat = feat.merge(vr.merge(msm, on='Outlet_ID'), on='Outlet_ID', how='left')
feat['var_ratio']            = feat['var_ratio'].fillna(0.5)
feat['months_since_new_max'] = feat['months_since_new_max'].fillna(12.0).clip(0,24)
feat['s_plateau'] = (
    0.5*(feat['var_ratio']<0.3).astype(float)
    + 0.5*(feat['months_since_new_max']/6).clip(0,1)
)
flag = (feat['var_ratio']<0.3) & (feat['months_since_new_max']>=4)
print(f'Strong plateau outlets: {flag.sum():,}')
print(feat[['var_ratio','months_since_new_max','s_plateau']].describe().round(3))

## 5. Constraint Score (4-component base)

`0.40 s_frontier + 0.30 s_sfa + 0.20 s_plateau + 0.10 s_peer_gap`

- **SFA fix (D1):** import from `pysfa.SFA` submodule; `fun='prod'`, `method='teJ'`; fallback = `s_frontier` (data-driven proxy)
- **SFA speed fix (D2):** fit on outlet-level `observed_max` (~20k rows) not 450k transaction rows
- No `^1.25` exponent (C6) | `has_valid_coordinates` removed (B4)
- Full POI 5-component variant is in **Section 10** (cell 20)

In [ ]:
# s_frontier — QR frontier residual (demand-unit anchored)
qrf = [c for c in ['Outlet_Size_Score','Outlet_Type_Score','Cooler_Count','mean_sku_count',
       'structural_capacity_score','catchment_density_score','poi_demand_score',
       'active_months','trend_ratio','cooler_adequacy'] if c in feat.columns]
qr = QuantileRegressor(quantile=0.9, alpha=0.0, solver='highs')
qr.fit(feat[qrf].fillna(0).to_numpy(), feat['observed_max_monthly_liters'].to_numpy())
feat['q90_frontier'] = np.maximum(qr.predict(feat[qrf].fillna(0).to_numpy()), feat['observed_max_monthly_liters'])
feat['s_frontier']   = ((feat['q90_frontier']-feat['observed_max_monthly_liters'])/(feat['q90_frontier']+1)).clip(0,1)
print(f"s_frontier: mean={feat['s_frontier'].mean():.3f}")

# s_sfa — Stochastic Frontier Analysis (outlet-level, ~20k rows — fast)
# Fallback = s_frontier (data-driven proxy, avoids uniform inflation from flat 0.5)
feat['s_sfa'] = feat['s_frontier'].copy()
sfa_source = 'fallback=s_frontier'
if SFA_AVAILABLE:
    try:
        from pysfa.SFA import SFA as _SFA
        if 'sfa_te' in feat.columns:
            feat.drop(columns=['sfa_te'], inplace=True)
        scols = ['Outlet_Size_Score','Outlet_Type_Score','Cooler_Count','mean_sku_count','target_jan_seasonality_score']
        sf_df = feat[['Outlet_ID','observed_max_monthly_liters'] + scols].copy()
        sf_df = sf_df[sf_df['observed_max_monthly_liters'] > 0].copy()
        sf_df['log_max'] = np.log(sf_df['observed_max_monthly_liters'] + 1)
        sfa_m = _SFA(y=sf_df['log_max'].values,
                     x=sf_df[scols].fillna(0).to_numpy(),
                     fun='prod', method='teJ')
        sfa_m.optimize()
        sf_df['sfa_te'] = sfa_m.get_technical_efficiency()
        mean_te = sf_df['sfa_te'].mean()
        if mean_te > 0.99 or bool((sf_df['sfa_te'] > 1.0).any()):
            raise ValueError(f'degenerate: mean_TE={mean_te:.3f} (BFGS converged to near-zero lambda on censored sales data)')
        feat = feat.merge(sf_df[['Outlet_ID','sfa_te']], on='Outlet_ID', how='left')
        feat['s_sfa'] = (1 - feat['sfa_te'].fillna(feat['s_frontier'])).clip(0, 1)
        sfa_source = f'fitted (mean_TE={mean_te:.3f})'
        print(f"SFA OK. Mean TE={mean_te:.3f}  |  s_sfa mean={feat['s_sfa'].mean():.3f}")
    except Exception as e:
        print(f'SFA skipped ({e})\n  → s_sfa = s_frontier (data-driven proxy)')

print(f"s_sfa source: {sfa_source}  |  mean={feat['s_sfa'].mean():.3f}")

# s_peer_gap
pg = feat.groupby(['Outlet_Type','Outlet_Size'])['observed_max_monthly_liters'].transform(lambda s: s.quantile(0.95)).fillna(feat['observed_max_monthly_liters'])
feat['s_peer_gap'] = ((pg-feat['observed_max_monthly_liters'])/(pg+1)).clip(0,1)

# Composite — B4: has_valid_coordinates REMOVED
feat['constraint_score'] = (
    0.40*feat['s_frontier']
    + 0.30*feat['s_sfa']
    + 0.20*feat['s_plateau']
    + 0.10*feat['s_peer_gap']
).clip(0,1)

print(feat['constraint_score'].describe().round(3))
print(feat.groupby('Outlet_Size')['constraint_score'].mean().round(3).sort_values())

## 6. GBM Models — Baseline & 90th-Percentile Frontier

In [ ]:
FEATURE_COLUMNS = [c for c in [
    'Outlet_Size','Outlet_Type',
    'Cooler_Count','Outlet_Size_Score','Outlet_Type_Score','structural_capacity_score',
    'observed_mean_monthly_liters','observed_max_monthly_liters','observed_std_monthly_liters',
    'active_months','mean_sku_count','max_sku_count','mean_transaction_lines',
    'january_mean_liters','january_max_liters','recent_3_month_mean_liters','recent_3_month_max_liters',
    'cv_liters','trend_ratio','peer_pct_by_type_size','peer_pct_by_size',
    'cooler_adequacy','sku_consistency','volume_per_cooler','volume_per_sku','dist_monthly_strength',
    'Latitude','Longitude',
    'outlet_count_1km','outlet_count_2km','outlet_count_5km',
    'same_type_cannibalisation_2km','nearest_outlet_distance_km','catchment_density_score',
    'poi_total_count_1km','poi_total_count_2km','poi_demand_score',
    'target_jan_seasonality_score','target_jan_holiday_count',
    's_frontier','s_sfa','s_plateau',
] if c in feat.columns]

CAT = ['Outlet_Size','Outlet_Type']
NUM = [c for c in FEATURE_COLUMNS if c not in CAT]

def make_pipe(params):
    return Pipeline([('pre', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT),
        ('num', 'passthrough', NUM),
    ])), ('mdl', HistGradientBoostingRegressor(**params, random_state=RANDOM_STATE))])

train_df = monthly[monthly['Monthly_Liters']>0].merge(feat[['Outlet_ID']+FEATURE_COLUMNS], on='Outlet_ID', how='inner')
Xtr, ytr = train_df[FEATURE_COLUMNS], train_df['Monthly_Liters']

base_pipe = make_pipe({'max_iter':180,'learning_rate':0.06,'l2_regularization':0.05})
base_pipe.fit(Xtr, ytr)

fron_pipe = make_pipe({'loss':'quantile','quantile':0.90,'max_iter':220,'learning_rate':0.05,'l2_regularization':0.05})
fron_pipe.fit(Xtr, ytr)

feat['observed_baseline_liters'] = base_pipe.predict(feat[FEATURE_COLUMNS])
feat['demand_frontier_liters']   = fron_pipe.predict(feat[FEATURE_COLUMNS])
print(f'Trained on {len(train_df):,} rows | {len(FEATURE_COLUMNS)} features')
print(f"Baseline mean={feat['observed_baseline_liters'].mean():.1f} | Frontier mean={feat['demand_frontier_liters'].mean():.1f}")

## 7. Final Predictions
Formula: `final = lower_bound + constraint_score × (peer_frontier − lower_bound)`

- No `^1.25` exponent (C6)
- Extra Large cap 4.0× (C5)

In [ ]:
lower_bound = feat[['observed_max_monthly_liters','january_max_liters','recent_3_month_max_liters']].max(axis=1).fillna(0)

pq90 = feat.groupby(['Outlet_Type','Outlet_Size'])['observed_max_monthly_liters'].transform(lambda s: s.quantile(0.90)).fillna(0)
tq85 = feat.groupby('Outlet_Type')['observed_max_monthly_liters'].transform(lambda s: s.quantile(0.85)).fillna(0)
sq85 = feat.groupby('Outlet_Size')['observed_max_monthly_liters'].transform(lambda s: s.quantile(0.85)).fillna(0)
feat['peer_frontier_liters'] = np.maximum.reduce([
    pq90.values, tq85.values*0.85, sq85.values,
    feat['demand_frontier_liters'].values, feat['q90_frontier'].values, lower_bound.values
])

gap          = (feat['peer_frontier_liters'] - lower_bound).clip(lower=0)
raw_potential = lower_bound + feat['constraint_score'] * gap

max_uplift   = feat['Outlet_Size'].map({'Unknown':2.0,'Small':3.0,'Medium':3.5,'Large':4.0,'Extra Large':4.0}).fillna(3.0)
peer_cap     = feat.groupby(['Outlet_Type','Outlet_Size'])['observed_max_monthly_liters'].transform(lambda s: s.quantile(0.98)).fillna(feat['observed_max_monthly_liters'].quantile(0.98))
soft_cap     = np.maximum(peer_cap*1.20, lower_bound*max_uplift)

raw_result   = np.minimum(np.maximum(raw_potential, lower_bound), soft_cap).clip(lower=0)
# Hard ratio cap: never exceed size-based multiplier × lower_bound (catches low-base edge cases)
raw_result   = np.minimum(raw_result, lower_bound * max_uplift)

# Rounding-violation fix: floor at observed max after rounding
feat['Maximum_Monthly_Liters'] = np.maximum(raw_result.round(3), feat['observed_max_monthly_liters'])

feat['uplift_ratio_vs_max'] = feat['Maximum_Monthly_Liters'] / (feat['observed_max_monthly_liters']+1)
print('Uplift distribution:')
print(feat['uplift_ratio_vs_max'].describe(percentiles=[.25,.5,.75,.9,.95,.99]).round(3))
print()
print(feat.groupby('Outlet_Size')[['uplift_ratio_vs_max']].agg(['count','mean','median']).round(3))

## 8. Submission & Validation

`Outlet_ID` column, 20 000 rows — per `Docs/problem.md` Deliverable 1.

In [ ]:
submission = feat[['Outlet_ID','Maximum_Monthly_Liters']].copy()
submission['Outlet_ID']               = submission['Outlet_ID'].astype(str)
submission['Maximum_Monthly_Liters']  = submission['Maximum_Monthly_Liters'].round(3)

checks = {
    'Row count == 20 000'  : len(submission) == 20_000,
    'Columns correct'      : list(submission.columns) == ['Outlet_ID','Maximum_Monthly_Liters'],
    'No null Outlet_ID'    : submission['Outlet_ID'].notna().all(),
    'No null predictions'  : submission['Maximum_Monthly_Liters'].notna().all(),
    'All predictions > 0'  : (submission['Maximum_Monthly_Liters'] > 0).all(),
    'All preds >= obs_max' : (feat['Maximum_Monthly_Liters'] >= feat['observed_max_monthly_liters']).all(),
    'No duplicate IDs'     : submission['Outlet_ID'].duplicated().sum() == 0,
    'Extreme uplift (>5x)' : (feat['uplift_ratio_vs_max'] > 5).sum() == 0,
}
for k,v in checks.items(): print(f"{'✓' if v else '✗'} {k}")

submission.to_csv(RESULTS_DIR/'smile_labs_predictions.csv', index=False)
print(f"\nSubmission saved: Results/smile_labs_predictions.csv  ({len(submission):,} rows)")

diag = feat[[
    'Outlet_ID','Outlet_Size','Outlet_Type','Distributor_ID',
    'observed_mean_monthly_liters','observed_max_monthly_liters',
    'q90_frontier','s_frontier','s_sfa','s_plateau','constraint_score',
    'peer_frontier_liters','catchment_density_score','poi_demand_score',
    'var_ratio','months_since_new_max',
    'Maximum_Monthly_Liters','uplift_ratio_vs_max',
]].copy()
diag.to_csv(GOLD_DIR/'prediction_diagnostics.csv', index=False)
print(f'Diagnostics saved: data/gold/prediction_diagnostics.csv')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

diag['uplift_ratio_vs_max'].clip(0, 5).hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Uplift distribution'); axes[0].set_xlabel('Uplift ×')

diag.groupby('Outlet_Size')['uplift_ratio_vs_max'].mean().sort_values().plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Mean uplift by size')

# Plot each constraint component individually to avoid pandas bin-shape mismatch
colors = ['steelblue', 'tomato', 'seagreen', 'darkorange']
labels = ['s_frontier', 's_sfa', 's_plateau', 'constraint_score']
for col, color, label in zip(labels, colors, labels):
    if col in diag.columns:
        axes[2].hist(diag[col].dropna(), bins=40, alpha=0.5, color=color, label=label)
axes[2].set_title('Constraint score components')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(RESULTS_DIR/'validation_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart → Results/validation_summary.png')

## 9. Comparison with Previous Pipeline Output

In [ ]:
old_path = RESULTS_DIR / 'smile_labs_predictions_full_20000.csv'
if not old_path.exists():
    print('Old predictions file not found — skipping comparison')
else:
    old = pd.read_csv(old_path)
    print(f"Old file columns: {list(old.columns)}")
    old_id_col   = old.columns[0]          # handles row_id OR Outlet_ID
    old_pred_col = old.columns[1]
    old['Outlet_ID'] = old[old_id_col].astype(str)

    cmp = diag[['Outlet_ID','Outlet_Size','Outlet_Type',
                'observed_max_monthly_liters','Maximum_Monthly_Liters',
                'uplift_ratio_vs_max','constraint_score']].merge(
        old[['Outlet_ID', old_pred_col]].rename(columns={old_pred_col: 'old_pred'}),
        on='Outlet_ID', how='inner'
    )
    cmp['old_uplift'] = cmp['old_pred'] / (cmp['observed_max_monthly_liters'] + 1)
    cmp['delta_pct']  = (cmp['Maximum_Monthly_Liters'] - cmp['old_pred']) / (cmp['old_pred'] + 1) * 100

    print(f"Matched outlets: {len(cmp):,}\n")
    stats = pd.DataFrame({
        'Old (01)': cmp['old_pred'].describe(percentiles=[.25,.5,.75,.9,.95,.99]),
        'New (11)': cmp['Maximum_Monthly_Liters'].describe(percentiles=[.25,.5,.75,.9,.95,.99]),
    })
    print(stats.round(1))
    print(f"\nMean delta: {cmp['delta_pct'].mean():+.2f}%  |  Median delta: {cmp['delta_pct'].median():+.2f}%")
    print(f"Higher: {(cmp['delta_pct']>0).sum():,}  |  Lower: {(cmp['delta_pct']<0).sum():,}")
    print(f"\nPOI coverage: {(diag['poi_demand_score']>0).sum():,} / {len(diag):,}")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].hist(cmp['old_pred'].clip(0,8000), bins=60, alpha=0.6, label='Old (01)', color='steelblue')
    axes[0].hist(cmp['Maximum_Monthly_Liters'].clip(0,8000), bins=60, alpha=0.6, label='New (11)', color='tomato')
    axes[0].set_title('Prediction distributions'); axes[0].set_xlabel('Max Monthly Liters'); axes[0].legend()
    cmp['delta_pct'].clip(-100,200).hist(bins=60, ax=axes[1], color='seagreen', edgecolor='white')
    axes[1].axvline(0, color='black', linewidth=1.2, linestyle='--')
    axes[1].set_title('Delta % (New − Old)'); axes[1].set_xlabel('Change %')
    plt.tight_layout(); plt.show()

## 10. Full POI Hot-swap + Re-predict

Loads `outlet_poi_features_full.csv` from `10_poi_enrichment_full.ipynb` (20,000/20,000 coverage).

**Changes vs base (Section 5–8):**
- POI columns replaced with full-country set (58 new columns)
- `s_poi` added as 5th constraint component (weight 0.10); other weights adjusted
- POI ceiling boost: peer soft cap × `(1.0 + 0.10 × poi_demand_score)` — bounded by hard ratio cap to prevent aggressive uplift
- Output: `Results/smile_labs_predictions_poi_full.csv` (base file unchanged)

**Prerequisites:** Run `10_poi_enrichment_full.ipynb` first.

In [ ]:
# ── Load ORIGINAL sparse-POI submission as true baseline ──────────────────
sparse_path = RESULTS_DIR / 'smile_labs_predictions.csv'
pred_before = pd.read_csv(sparse_path).rename(columns={'Maximum_Monthly_Liters': 'pred_sparse_poi'})
pred_before['Outlet_ID'] = pred_before['Outlet_ID'].astype(str)
print(f'Sparse baseline loaded: {len(pred_before):,} rows from {sparse_path.name}')

# ── Load full-coverage POI from notebook 10 ───────────────────────────────
new_poi_path = GOLD_DIR / 'outlet_poi_features_full.csv'
if not new_poi_path.exists():
    raise FileNotFoundError('Run 10_poi_enrichment_full.ipynb first.')

new_poi = pd.read_csv(new_poi_path)
new_poi['Outlet_ID'] = new_poi['Outlet_ID'].astype(str)
print(f'POI coverage after:  {(new_poi["poi_demand_score"] > 0).sum():,} / {len(new_poi):,}')

# Drop old POI columns and merge full-coverage ones
old_poi_cols = [c for c in feat.columns if c.startswith('poi_')]
feat.drop(columns=old_poi_cols, inplace=True)
new_poi_cols = [c for c in new_poi.columns if c != 'Outlet_ID']
feat = feat.merge(new_poi[['Outlet_ID'] + new_poi_cols], on='Outlet_ID', how='left')
for c in new_poi_cols:
    feat[c] = pd.to_numeric(feat[c], errors='coerce').fillna(0)
print(f'New POI columns: {len(new_poi_cols)} | feat shape: {feat.shape}')

# ── Re-run QR frontier ────────────────────────────────────────────────────
qrf2 = [c for c in ['Outlet_Size_Score','Outlet_Type_Score','Cooler_Count','mean_sku_count',
        'structural_capacity_score','catchment_density_score','poi_demand_score',
        'active_months','trend_ratio','cooler_adequacy'] if c in feat.columns]
qr2 = QuantileRegressor(quantile=0.9, alpha=0.0, solver='highs')
qr2.fit(feat[qrf2].fillna(0).to_numpy(), feat['observed_max_monthly_liters'].to_numpy())
feat['q90_frontier'] = np.maximum(qr2.predict(feat[qrf2].fillna(0).to_numpy()), feat['observed_max_monthly_liters'])
feat['s_frontier']   = ((feat['q90_frontier'] - feat['observed_max_monthly_liters']) / (feat['q90_frontier'] + 1)).clip(0, 1)

# ── SFA (degenerate-safe, same logic as cell 10) ──────────────────────────
feat['s_sfa'] = feat['s_frontier'].copy()   # fallback = s_frontier (data-driven proxy)
sfa_source2 = 'fallback=s_frontier'
if SFA_AVAILABLE:
    try:
        from pysfa.SFA import SFA as _SFA
        if 'sfa_te' in feat.columns:
            feat.drop(columns=['sfa_te'], inplace=True)
        scols = ['Outlet_Size_Score','Outlet_Type_Score','Cooler_Count','mean_sku_count','target_jan_seasonality_score']
        sf_df = feat[['Outlet_ID','observed_max_monthly_liters'] + scols].copy()
        sf_df = sf_df[sf_df['observed_max_monthly_liters'] > 0].copy()
        sf_df['log_max'] = np.log(sf_df['observed_max_monthly_liters'] + 1)
        sfa_m = _SFA(y=sf_df['log_max'].values,
                     x=sf_df[scols].fillna(0).to_numpy(),
                     fun='prod', method='teJ')
        sfa_m.optimize()
        sf_df['sfa_te'] = sfa_m.get_technical_efficiency()
        mean_te = sf_df['sfa_te'].mean()
        if mean_te > 0.99 or bool((sf_df['sfa_te'] > 1.0).any()):
            raise ValueError(f'degenerate: mean_TE={mean_te:.3f}')
        feat = feat.merge(sf_df[['Outlet_ID','sfa_te']], on='Outlet_ID', how='left')
        feat['s_sfa'] = (1 - feat['sfa_te'].fillna(feat['s_frontier'])).clip(0, 1)
        sfa_source2 = f'fitted (mean_TE={mean_te:.3f})'
        print(f"SFA OK. Mean TE={mean_te:.3f}  |  s_sfa mean={feat['s_sfa'].mean():.3f}")
    except Exception as e:
        print(f'SFA skipped ({e})\n  → s_sfa = s_frontier (data-driven proxy)')

print(f"s_sfa source: {sfa_source2}  |  mean={feat['s_sfa'].mean():.3f}")

# ── Peer gap ──────────────────────────────────────────────────────────────
pg2 = feat.groupby(['Outlet_Type','Outlet_Size'])['observed_max_monthly_liters'].transform(
    lambda s: s.quantile(0.95)).fillna(feat['observed_max_monthly_liters'])
feat['s_peer_gap'] = ((pg2 - feat['observed_max_monthly_liters']) / (pg2 + 1)).clip(0, 1)

# ── Constraint score: POI as direct 5th component (D4, weight 0.10) ───────
feat['s_poi'] = feat['poi_demand_score'].clip(0, 1)
feat['constraint_score'] = (
    0.35 * feat['s_frontier']
    + 0.30 * feat['s_sfa']
    + 0.15 * feat['s_plateau']
    + 0.10 * feat['s_peer_gap']
    + 0.10 * feat['s_poi']
).clip(0, 1)
print(f"\nConstraint score  mean={feat['constraint_score'].mean():.3f}  std={feat['constraint_score'].std():.3f}")
print(feat.groupby('Outlet_Size')['constraint_score'].mean().round(3).sort_values())

# ── Re-train frontier GBM ─────────────────────────────────────────────────
FEATURE_COLUMNS2 = list(dict.fromkeys(
    [c for c in FEATURE_COLUMNS if c in feat.columns]
    + [c for c in ['poi_total_count_1km','poi_total_count_2km','poi_demand_score','s_poi'] if c in feat.columns]
))
CAT2 = ['Outlet_Size','Outlet_Type']
NUM2 = [c for c in FEATURE_COLUMNS2 if c not in CAT2]
fron_pipe2 = Pipeline([('pre', ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT2),
    ('num', 'passthrough', NUM2),
])), ('mdl', HistGradientBoostingRegressor(loss='quantile', quantile=0.90,
      max_iter=220, learning_rate=0.05, l2_regularization=0.05, random_state=RANDOM_STATE))])
train2 = monthly[monthly['Monthly_Liters']>0].merge(feat[['Outlet_ID']+FEATURE_COLUMNS2], on='Outlet_ID', how='inner')
fron_pipe2.fit(train2[FEATURE_COLUMNS2], train2['Monthly_Liters'])
feat['demand_frontier_liters'] = fron_pipe2.predict(feat[FEATURE_COLUMNS2])
print(f'Frontier re-trained | {len(FEATURE_COLUMNS2)} features')

# ── Final predictions ─────────────────────────────────────────────────────
lb2   = feat[['observed_max_monthly_liters','january_max_liters','recent_3_month_max_liters']].max(axis=1).fillna(0)
pq90b = feat.groupby(['Outlet_Type','Outlet_Size'])['observed_max_monthly_liters'].transform(lambda s: s.quantile(0.90)).fillna(0)
tq85b = feat.groupby('Outlet_Type')['observed_max_monthly_liters'].transform(lambda s: s.quantile(0.85)).fillna(0)
sq85b = feat.groupby('Outlet_Size')['observed_max_monthly_liters'].transform(lambda s: s.quantile(0.85)).fillna(0)
feat['peer_frontier_liters'] = np.maximum.reduce([
    pq90b.values, tq85b.values*0.85, sq85b.values,
    feat['demand_frontier_liters'].values, feat['q90_frontier'].values, lb2.values])

# D5: POI boost relaxes the peer-quantile soft cap only (factor 0.10).
# The hard ratio cap (lb2 * mu2) below is the absolute ceiling and intentionally
# overrides the soft cap to prevent aggressive uplift on low-base outlets.
poi_ceiling_boost = 1.0 + 0.10 * feat['poi_demand_score']
gap2  = (feat['peer_frontier_liters'] - lb2).clip(lower=0)
raw2  = lb2 + feat['constraint_score'] * gap2
mu2   = feat['Outlet_Size'].map({'Unknown':2.0,'Small':3.0,'Medium':3.5,'Large':4.0,'Extra Large':4.0}).fillna(3.0)
cap2  = np.maximum(
    feat.groupby(['Outlet_Type','Outlet_Size'])['observed_max_monthly_liters'].transform(
        lambda s: s.quantile(0.98)).fillna(feat['observed_max_monthly_liters'].quantile(0.98))
    * 1.20 * poi_ceiling_boost,
    lb2 * mu2)
raw_result = np.minimum(np.maximum(raw2, lb2), cap2).clip(lower=0)
raw_result = np.minimum(raw_result, lb2 * mu2)   # hard ratio cap: absolute ceiling to prevent aggressive boosting
feat['Maximum_Monthly_Liters'] = np.maximum(raw_result.round(3), feat['observed_max_monthly_liters'])
feat['uplift_ratio_vs_max'] = feat['Maximum_Monthly_Liters'] / (feat['observed_max_monthly_liters'] + 1)

# ── Validation ────────────────────────────────────────────────────────────
n_extreme = int((feat['uplift_ratio_vs_max'] > 5).sum())
checks = {
    'All preds >= obs_max' : bool((feat['Maximum_Monthly_Liters'] >= feat['observed_max_monthly_liters']).all()),
    'No NaN predictions'   : bool(feat['Maximum_Monthly_Liters'].notna().all()),
    'Extreme uplift (>5x)' : n_extreme,
}
for k, v in checks.items():
    ok = v is True or v == 0
    print(f"{'✓' if ok else '✗'} {k}: {v}")
print(f"  s_sfa source: {sfa_source2}")

# ── Save ──────────────────────────────────────────────────────────────────
sub2 = feat[['Outlet_ID','Maximum_Monthly_Liters']].copy()
sub2['Outlet_ID']              = sub2['Outlet_ID'].astype(str)
sub2['Maximum_Monthly_Liters'] = sub2['Maximum_Monthly_Liters'].round(3)
out_path = RESULTS_DIR / 'smile_labs_predictions_poi_full.csv'
sub2.to_csv(out_path, index=False)
print(f'\nSaved: Results/smile_labs_predictions_poi_full.csv ({len(sub2):,} rows)')

# ── Compare sparse baseline (from disk) vs full POI ──────────────────────
cmp2 = feat[['Outlet_ID','Outlet_Size','observed_max_monthly_liters',
             'Maximum_Monthly_Liters','uplift_ratio_vs_max']].merge(pred_before, on='Outlet_ID')
cmp2['delta_pct'] = (cmp2['Maximum_Monthly_Liters'] - cmp2['pred_sparse_poi']) / (cmp2['pred_sparse_poi'] + 1) * 100

print('\n── Sparse POI baseline vs Full POI ──────────────────')
print(pd.DataFrame({
    'Sparse POI baseline': cmp2['pred_sparse_poi'].describe(percentiles=[.25,.5,.75,.9,.95,.99]),
    'Full POI (this run)': cmp2['Maximum_Monthly_Liters'].describe(percentiles=[.25,.5,.75,.9,.95,.99]),
}).round(1))
print(f'\nMean delta:  {cmp2["delta_pct"].mean():+.2f}%')
print(f'Higher: {(cmp2["delta_pct"]>0).sum():,}  |  Lower: {(cmp2["delta_pct"]<0).sum():,}')
print(f'\nMean uplift by size:')
print(cmp2.groupby('Outlet_Size')['uplift_ratio_vs_max'].mean().round(3).sort_values())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(cmp2['pred_sparse_poi'].clip(0,8000), bins=60, alpha=0.6, label='Sparse POI baseline', color='steelblue')
axes[0].hist(cmp2['Maximum_Monthly_Liters'].clip(0,8000), bins=60, alpha=0.6, label='Full POI', color='tomato')
axes[0].set_title('Sparse baseline vs Full POI'); axes[0].legend()
cmp2['delta_pct'].clip(-100, 200).hist(bins=60, ax=axes[1], color='seagreen', edgecolor='white')
axes[1].axvline(0, color='black', linewidth=1.2, linestyle='--')
axes[1].set_title('Delta % (Full POI − Sparse baseline)')
for col, color, label in zip(['s_frontier','s_sfa','s_plateau','s_poi','constraint_score'],
                              ['steelblue','tomato','seagreen','darkorange','black'],
                              ['s_frontier','s_sfa','s_plateau','s_poi','constraint_score']):
    if col in feat.columns:
        axes[2].hist(feat[col].dropna(), bins=40, alpha=0.45, color=color, label=label)
axes[2].set_title('Constraint components'); axes[2].legend(fontsize=7)
plt.tight_layout(); plt.show()